# Современные методы анализа данных и машинного обучения, БИ

## НИУ ВШЭ, 2025-26 учебный год

### Задание 1

Проанализируйте набор данных и ответьте на следующие вопросы:

1) Какой процент наблюдений выходит за 5 и 95 перцентиль?

2) Какому закону распределения отвечают данные? На основе чего вы так решили?

[Ссылка на данные](https://drive.google.com/file/d/1fc8BGSjDiwbZW6rcr3mMSwAjU2T_BpGW/view?usp=sharing)

Описание данных:

* `metrics` — значение определенной метрики в сервисе для пользователя (например, количество кликов).

*Вместе с ответом на задание приложить код*

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

df = pd.read_csv('test_1_1.csv', sep=';')
m = df['metrics']
p5, p95 = m.quantile(0.05), m.quantile(0.95)

In [9]:
p5, p95

(np.float64(1.0), np.float64(52.0))

In [2]:
(m < p5).mean()*100

np.float64(0.0)

In [3]:
(m > p95).mean()*100

np.float64(4.987306046493438)

In [4]:
((m < p5) | (m > p95)).mean()*100

np.float64(4.987306046493438)

In [5]:
m.mean(), m.median(), m.mode()[0]

(np.float64(10.146275408276), 2.0, np.int64(1))

In [6]:
m.var(), m.skew(), m.kurtosis()

(511.99823268544515,
 np.float64(5.231829907292705),
 np.float64(51.91321889673423))

In [7]:
vals, cnts = np.unique(m, return_counts=True)
slope, _, r, _, _ = linregress(np.log(vals), np.log(cnts/len(m)))
slope

np.float64(-2.4216822451409348)

In [8]:
r**2

np.float64(0.9067199104918269)

p5 = 1 , p95 = 52

Ниже p5 0% (минимум в данных = 1, он же и есть p5)

Выше p95 4.99%

Итого за границами ≈5%

2) Степенному, так как:

Данные дискретные, мода=1, медиана=2, среднее=10 — огромный разрыв, сильная правая асимметрия (skew=5.23)

Дисперсия (512) сильно больше среднего (10), то есть пуассон исключён сразу

В log-log координатах частота от значения ложится почти на прямую (slope=−2.42, R^2 =0.91)

### Задание 2

Определите, применим ли t-критерий Стьюдента для сравнения двух представленных выборок.

Если применение t-критерия Стьюдента возможно, выполните расчет.

Если сравнение с помощью t-критерия Стьюдента невозможно, укажите, какой критерий следует использовать вместо него, и обоснуйте свой выбор.

Ваши ответы аргументируйте.

[Ссылка на данные](https://drive.google.com/file/d/1blvC6TEBWghhVMexjOdoGEjzhZZh3wSy/view?usp=sharing)

Описание данных:

* `variant` — вариация в А/B тесте (например, группа А — сайт без изменений; группа B — сайт с изменениями)
* `metrics` — значение определенной метрики в сервисе для пользователя (например, количество кликов).

*Вместе с ответом на задание приложить код*

In [10]:
from scipy import stats

df = pd.read_csv('test2.csv', sep=';')
a = df[df['variant'] == 'A']['metrics']
b = df[df['variant'] == 'B']['metrics']

In [11]:
a.mean()

np.float64(10.099561961892842)

In [12]:
a.skew()

np.float64(5.989602976630683)

In [13]:
b.mean()

np.float64(10.33313665071352)

In [14]:
b.skew()

np.float64(4.971141697389426)

In [19]:
stat_a, p_a = stats.shapiro(a.sample(5000, random_state=42))
stat_b, p_b = stats.shapiro(b.sample(5000, random_state=42))
print(p_a, p_b)

1.8007893312897872e-83 1.5040835122059317e-81


In [17]:
stat, p = stats.mannwhitneyu(a, b, alternative='two-sided')
print(stat, p)

4195358346.0 2.7311182624546524e-05


T-критерий неприменим, так как нарушено условие нормальности. Shapiro-Wilk даёт p≈0 в обеих группах, skewness у а=5.99, у b=4.97, то есть распределения сильно скошены вправо

Вместо него используем критерий Манна-Уитни — это непараметрический аналог t-теста, который не требует нормальности и работает со скошенными распределениями. Он сравнивает не средние, а ранги значений

Результат: p≈0 - различия между группами A и B статистически значимы

### Задание 3

Вы работаете в стартапе, который разрабатывает приложение для грибников. Приложение с помощью компьютерного зрения определяет, съедобен ли гриб. Сейчас алгоритм работает правильно только в 50% случаев, то есть его точность не лучше случайного угадывания.

К вам приходит ML-инженер и заявляет, что улучшил алгоритм. Он просит премию и показывает результаты тестирования: на 120 грибах его новая модель дала правильный ответ в 79 случаях.

На основе этих данных вам необходимо:
1. Обосновать, стоит ли выплачивать инженеру премию, с помощью статистической проверки гипотез. Сформулируйте нулевую и альтернативную гипотезы, укажите используемый критерий, уровень значимости и сделайте вывод.
2. Определить, готова ли новая версия алгоритма к выпуску. Учтите не только статистическую значимость, но и практическую применимость.

Ваши ответы аргументируйте.

*Вместе с ответом на задание приложить код*

1. Проверка гипотез

H0: точность модели = 0.5 (не лучше случайного)
H1: точность модели > 0.5 (модель улучшилась)
Критерий: биномиальный тест (данные — счётные, одна выборка, проверяем долю)
Уровень значимости: α = 0.05

In [22]:
n = 120
k = 79
p0 = 0.5

res = stats.binomtest(k, n, p0, alternative='greater')
print("точность = ", k/n)
print("p-value = ", res.pvalue)

точность =  0.6583333333333333
p-value =  0.0003333619485017593


p-value = 0.00033 < 0.05, следовательно отвергаем H0. Улучшение статистически значимо, премию выплатить стот


In [24]:
ci = res.proportion_ci(confidence_level=0.95)
ci.low, ci.high

(0.580545266901544, 1.0)

2. Готова ли модель к выпуску — нет
Точность 65.8%, 95% ДИ: (58%, 100%). Нижняя граница доверительного интервала всего 58%. Для приложения про грибы это критично (каждый третий гриб определяется неверно). Ошибка может стоить пользователю жизни, здесь нужна точность 95%+. Статистически значимо не значит практически применимо

### Задание 4

Перед вами представлен датасет `Employee_satisfaction.csv`, содержащий следующую информацию:

[Ссылка на данные](https://drive.google.com/file/d/1XSuJH8vy6KFICYAkRc7qMvPrMKO2lZ6o/view?usp=sharing)

Описание данных:

* `Emp ID` - порядковый номер сотрудника
* `satisfaction_level` - самооценка уровня удовлетворенности работой
* `last_evaluation` - оценка на последнем перформанс-ревью
* `number_project` - количество проектов, над которыми работает сотрудник
* `average_montly_hours` - среднее количество часов, отработанных сотрудником в месяц
* `time_spend_company` - количество лет, которые сотрудник работает в компании
* `Work_accident` - попадал ли сотрудник в какой-то инцидент на работе (1 - да, 0 - нет)
* `promotion_last_5years` - получал ли сотрудник повышение за последние 5 лет (1 - да, 0 - нет)
* `dept` - отдел, где работает сотрудник
* `salary` - уровень зарплаты сотрудника (e.g., low, medium)

Очистите данные от пропущенных значений. Проанализировав взаимосвязь уровня удовлетворенности работой с остальными признаками, ответьте на следующие вопросы:
1. Есть ли какая-то статистически значимая связь между удовлетворенностью и оценкой на ревью?
2. Есть ли какая-то статистически значимая связь между удовлетворенностью и зарплатой? При ответе на вопрос сформируйте из переменной `salary` новую переменную, содержащую два значения: низкую зарплату `low` и все остальные
3. Есть ли связь между удовлетворенностью работой и средним количеством рабочих часов в месяц?

Дайте развернутые ответы с указанием результата, описанием вывода и способом его получения.

*Вместе с ответом на задание приложить код*

In [26]:
df = pd.read_csv('Employee_satisfaction.csv').dropna()
sat = df['satisfaction_level']

In [30]:
r, p = stats.pearsonr(sat, df['last_evaluation'])
print(r, p)


0.10502121397148488 4.7043115582916e-38


1. Использовали корреляцию Пирсона, обе переменные непрерывные. r=0.105, p≈0. Связь статистически значима (p<0.05), но слабая. Чем выше оценка на ревью, тем чуть выше удовлетворённость, но практически это почти не ощущается

In [28]:
df['salary_bin'] = df['salary'].apply(lambda x: 'low' if x == 'low' else 'other')
low = df[df['salary_bin'] == 'low']['satisfaction_level']
other = df[df['salary_bin'] == 'other']['satisfaction_level']
stat, p2 = stats.mannwhitneyu(low, other, alternative='two-sided')
print(low.mean(),other.mean(),p2)

0.6007531437944232 0.6243368475855785 9.628310958572507e-08


2. Создала бинарную переменную low vs other (medium+high). Использовала Mann-Whitney — сравниваем две группы, нормальность не гарантирована. Среднее удовлетворённости: low=0.6, other=0.62. p≈0 — связь статистически значима, но разница оч маленькая (0.02). На практике зарплата в данном датасете почти не объясняет удовлетворённость

In [29]:
r3, p3 = stats.pearsonr(sat, df['average_montly_hours'])
print(r3, p3)

-0.020048113219472995 0.014075035446899918


3. Корреляция Пирсона: r=−0.02, p=0.014. Связь статистически значима, но r почти ноль, это означает что связи практически нет. Количество рабочих часов не объясняет удовлетворённость сотрудников

### Задание 5

В компании решили ввести новую систему обучения для сотрудников и оценить ее эффективность с помощью A/B-теста. Результаты представлены в датасете `hr_ab_test_dataset.csv`.

[Ссылка на данные](https://drive.google.com/file/d/1DYGyX1IpOrczc1dJ4g98lybPXL3oWnN5/view?usp=sharing)

Описание данных:

* `Group` - группа A (сотрудники, прошедшие обучение), группа B (сотрудники, не прошедшие обучение).
* `Age` - возраст сотрудника.
* `Performance score` - метрика "Оценка производительности".

Гипотеза - обучение положительно влияет на оценку производительности.

На основе анализа результатов теста определите, какая из групп (A или B) продемонстрировала лучшие результаты. Какие статистические методы (критерии) вы использовали для принятия этого решения? Сформулируйте выводы относительно эффективности новой системы обучения, а также установите, существует ли связь между достигнутыми результатами и возрастом участников. Ответ аргументируйте.

*Вместе с ответом на задание приложить код*

In [31]:
df = pd.read_csv('hr_ab_test_dataset.csv')
a = df[df['Group'] == 'A']['Performance score']
b = df[df['Group'] == 'B']['Performance score']

In [32]:
a.mean()

np.float64(3.9480767413029527)

In [33]:
b.mean()

np.float64(3.0111522935249626)

1. Группа А продемонстрировала лучшие рузельтаты:

Группа A (обучение): mean=3.95, группа B (без обучения): mean=3.01.
Разница почти в 1 балл

In [34]:
stat_a, p_a = stats.shapiro(a)
stat_b, p_b = stats.shapiro(b)
print(p_a, p_b)

0.6551676754214469 0.08525602813445946


Сначала проверили нормальность через Shapiro-Wilk: A p=0.655, B p=0.085 — обе группы нормальные (p>0.05). Формально можно t-тест, но использовали Mann-Whitney как более устойчивый вариант при небольших выборках (n=100 в каждой). Тест односторонний, так как гипотеза направленная: A > B

In [35]:
stat, p = stats.mannwhitneyu(a, b, alternative='greater')
print(stat, p)

9170.0 1.1250092973810728e-24


p≈0, значит различие статистически значимо. Обучение эффективно: группа A показала производительность на 31% выше чем группа B. Систему обучения стоит внедрять

In [36]:
r, p_r = stats.pearsonr(df['Age'], df['Performance score'])
print(r, p_r)

-0.0901314165338109 0.20435330873057125


Корреляция Пирсона: r=−0.09, p=0.204. p>0.05 - значит связь статистически незначима. Возраст не влияет на результат, обучение одинаково эффективно для сотрудников любого возраста